In [1]:
import os
import csv
from bs4 import BeautifulSoup

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
input_folder = '../02_raw/html/'   # folder with saved .html files
output_folder = '../02_raw/csv/'    # folder for CSV output

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# ------------------------------------------------------------
# FUNCTION TO PARSE A SINGLE HTML FILE
# ------------------------------------------------------------
def parse_html_file(filepath):
    """
    Parse a single HTML file containing benchmark data and write extracted data to a CSV file.

    The HTML is expected to contain:
        - an <h3> tag with id="results" – its first text content gives the packet name,
        - an <h5> tag – its first text content gives the benchmark name,
        - a <div> with class "div_table_first_row" – defines column headers,
        - multiple <div> elements with class "div_table_row" – each contains data cells.

    The last cell of a row may contain a '+/-' pattern; the value before '+' is taken as the main value,
    and the part after '+/-' becomes an 'Intervals' column.

    Parameters
    ----------
    filepath : str
        Path to the input HTML file.

    Returns
    -------
    str or None
        Path to the created CSV file if parsing was successful and at least one data row exists,
        otherwise None.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        html = f.read()

    soup = BeautifulSoup(html, 'html.parser')

    results_section = soup.find('h3', id="results")
    model_section = soup.find('h5')
    first_row = soup.find(class_="div_table_first_row")

    if not results_section or not model_section or not first_row:
        print(f"Skipped {filepath}: missing required blocks (h3#results, h5, .div_table_first_row)")
        return None

    # Important: In your files, h3#results contains the Packet (e.g., GROMACS_2020.1)
    # and h5 contains the Benchmark (e.g., Water_Benchmark)
    packet_raw = results_section.contents[0] if results_section.contents else "Unknown"
    packet_name = packet_raw.replace(' ', '_')

    benchmark_raw = model_section.contents[0] if model_section.contents else "Unknown"
    benchmark_name = benchmark_raw.replace(' ', '_').replace(':', '')

    header_text = first_row.text.splitlines()
    original_headers = [cell.strip() for cell in header_text if cell.strip()]

    new_headers = ['Packet', 'Benchmark'] + original_headers + ['Intervals']

    data_rows = soup.find_all(class_="div_table_row")
    rows = [new_headers]

    for row_div in data_rows:
        cells = row_div.find_all('div')
        table_values = [cell.text.strip().replace('< ', '') for cell in cells]

        last_cell = table_values[-1]
        if '+/-' in last_cell:
            parts = last_cell.split('+/-')
            value = parts[0].strip()
            interval = parts[1].strip()
        else:
            value = last_cell
            interval = '0'
        table_values[-1] = value

        row_data = [packet_name, benchmark_name] + table_values + [interval]
        rows.append(row_data)

    if len(rows) > 1:
        # Original filename format: Packet-Benchmark.csv
        out_filename = f"{packet_name}-{benchmark_name}.csv"
        out_path = os.path.join(output_folder, out_filename)
        with open(out_path, 'w', newline='', encoding='utf-8') as f_out:
            writer = csv.writer(f_out)
            writer.writerows(rows)
        print(f"Created: {out_path}")
        return out_path
    else:
        print(f"No data rows in {filepath}")
        return None

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
if not os.path.exists(input_folder):
    print(f"Folder {input_folder} not found. Please create it and place .html files inside.")
else:
    html_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.html')]
    if not html_files:
        print(f"There are no .html files in folder {input_folder}.")
    else:
        for filename in html_files:
            filepath = os.path.join(input_folder, filename)
            print(f"Processing: {filename}")
            parse_html_file(filepath)

Processing: LAMMPS Molecular Dynamics Simulator Benchmark - OpenBenchmarking.org.22.20.html
Created: ../02_raw/csv/LAMMPS_Molecular_Dynamics_Simulator_23Jun2022-Model_20k_Atoms.csv
